In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
from training.models import get_resnet18, get_vit_tiny
from training.trainer import Trainer
from training.logger import CometLogger

/root/Course_work_HSE_2026/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
experiment_name = "resnet18_baseline_no_aug"   # уникальное имя
seed = 42
num_epochs = 30
batch_size = 64
learning_rate = 1e-3
num_classes = 102   # для Oxford Flowers 102
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [1]:
from local_datasets import OxfordFlowers102

In [4]:
flowers = OxfordFlowers102(root='./data', transform=None, download=True)

In [5]:
from utils.transforms import train_transform, val_test_transform

In [22]:
flowers.train_dataset.transform = train_transform

In [23]:
flowers.val_dataset.transform = val_test_transform
flowers.test_dataset.transform = val_test_transform

In [26]:
from training.models import get_resnet18, get_vit_tiny

In [ ]:
model = get_vit_tiny(num_classes=num_classes, pretrained=True)

In [13]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5)

In [15]:
train_loader = DataLoader(flowers.train_dataset, batch_size=batch_size, shuffle=True,
                              num_workers=4, pin_memory=True)
val_loader = DataLoader(flowers.val_dataset, batch_size=batch_size, shuffle=False,
                            num_workers=4, pin_memory=True)

In [18]:
comet_logger = CometLogger(
    project_name="augmentation-comparison",
    experiment_name=experiment_name,
    api_key="MbL2psOHT82Uc7ML5Cd7TSvmR",  # можно указать явно

)
# Логируем гиперпараметры
comet_logger.log_parameters({
    "model": "resnet18",
    "num_classes": num_classes,
    "batch_size": batch_size,
    "learning_rate": learning_rate,
    "optimizer": "Adam",
    "scheduler": "ReduceLROnPlateau",
    "seed": seed,
    "augmentation": "none",
})

COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: torch.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/-3767/augmentation-comparison/6616d4d741634101bfffa07da78a30cc



In [ ]:
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    experiment_name=experiment_name,
    num_epochs=num_epochs,
    log_dir="logs",               # папка для локальных логов
    comet_logger=comet_logger,
    seed=seed,
)

In [25]:
trainer.fit()

Training started for 30 epochs on device cuda
Epoch 1: new best model saved with val_acc 0.3039
Epoch [1/30] Train Loss: 3.3627 Acc: 0.3078 Top5: 0.4951 | Val Loss: 3.3063 Acc: 0.3039 Top5: 0.5961
Epoch 2: new best model saved with val_acc 0.6157
Epoch [2/30] Train Loss: 1.2594 Acc: 0.7618 Top5: 0.9441 | Val Loss: 1.6309 Acc: 0.6157 Top5: 0.8578
Epoch 3: new best model saved with val_acc 0.7765
Epoch [3/30] Train Loss: 0.5327 Acc: 0.8990 Top5: 0.9892 | Val Loss: 0.9731 Acc: 0.7765 Top5: 0.9461
Epoch [4/30] Train Loss: 0.2500 Acc: 0.9627 Top5: 0.9980 | Val Loss: 0.9226 Acc: 0.7765 Top5: 0.9412
Epoch 5: new best model saved with val_acc 0.8000
Epoch [5/30] Train Loss: 0.1326 Acc: 0.9873 Top5: 1.0000 | Val Loss: 0.8157 Acc: 0.8000 Top5: 0.9529
Epoch 6: new best model saved with val_acc 0.8176
Epoch [6/30] Train Loss: 0.0900 Acc: 0.9873 Top5: 1.0000 | Val Loss: 0.7964 Acc: 0.8176 Top5: 0.9461
Epoch 7: new best model saved with val_acc 0.8451
Epoch [7/30] Train Loss: 0.0614 Acc: 0.9931 Top5

COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : resnet18_baseline_no_aug
COMET INFO:     url                   : https://www.comet.com/-3767/augmentation-comparison/6616d4d741634101bfffa07da78a30cc
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     train_accuracy [30]      : (0.307843137254902, 1.0)
COMET INFO:     train_batch_loss [480]   : (0.001739064115099609, 4.896495819091797)
COMET INFO:     train_loss [30]          : (0.004350925500358583, 3.36266362620335)
COMET INFO:     train_top5_accuracy [30] : (0.4950980392156863, 1.0)
COMET INFO:     val_accuracy [30]        : (0.30392156862745096, 0.9088235294117647)
COMET INFO:     val_batch_loss [480]     : (0.0596587173640728, 5.434725761

Epoch [30/30] Train Loss: 0.0044 Acc: 1.0000 Top5: 1.0000 | Val Loss: 0.3665 Acc: 0.9078 Top5: 0.9814


COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: torch.
COMET INFO: Please wait for metadata to finish uploading (timeout is 3600 seconds)
COMET INFO: Uploading 1 metrics, params and output messages


Training finished.
